<a href="https://colab.research.google.com/github/oooinr4018-web/-1/blob/main/%EC%8B%9C%EC%8A%A4%ED%85%9C%EA%B5%AC%EC%B6%95%EC%A0%84_%EC%88%98%EC%A0%95%EA%B3%BC%EC%A0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- 예상보상금 병합 단계에서 문제 발생

- 일부 조합이 최종예측보상액 = NaN이 되는 오류 발생

1) 예상보상금보완

기존 '학교급+장소+시간+형태=가 일치하는 경우만 보상금을 가져왔는데, 없는 조합은 NaN이었음.

-> 4개 조건 일치 -> 학교급+장소+형태 -> 학교급+시간+형태 -> 학교급+형태 -> 학교급+장소 순으로 유사사례 중앙값을 넣어 NaN 모두 제거

2) TOPSIS 재계산

TOPSIS는 X1 위험확률, X2 예측보상금, X3 사고빈도 사용. X2가 앞에서 바뀌었으므로 TOPSIS 다시 계산.

3) SHAP에는 영향 X

SHAP은 위험등급_모델에서 나온 값이고, 이 수정 작업은 예측보상금이 바뀐거라 SHAP에 영향 X



In [9]:
import os

os.listdir("/content")

['.config',
 '통합_사고예방_데이터_초안(배포용 X).csv',
 '.ipynb_checkpoints',
 '위험도_위험등급_SHAP_최종결과.csv',
 '예상보상금_결과.csv',
 'sample_data']

In [10]:
import pandas as pd

pred = pd.read_csv("예상보상금_결과.csv", encoding="utf-8-sig")
risk = pd.read_csv("위험도_위험등급_SHAP_최종결과.csv", encoding="utf-8-sig")
draft = pd.read_csv("통합_사고예방_데이터_초안(배포용 X).csv", encoding="utf-8-sig")

print("예상보상금:", pred.shape)
print("위험도:", risk.shape)
print("초안:", draft.shape)

print("\n=== 예상보상금 컬럼 ===")
print(pred.columns.tolist())

print("\n=== 위험도 컬럼 ===")
print(risk.columns.tolist())

print("\n=== 초안 컬럼 ===")
print(draft.columns.tolist())

예상보상금: (143895, 6)
위험도: (2313, 16)
초안: (142670, 45)

=== 예상보상금 컬럼 ===
['지역', '학교급', '사고장소', '사고시간', '사고형태', '최종예측보상액']

=== 위험도 컬럼 ===
['학교급', '사고장소', '사고형태', '사고건수', '평균보상액', '빈도율', '빈도_점수', '심도_점수', '위험도_점수', '위험등급', 'SHAP_주요요인1', 'SHAP_주요요인2', 'SHAP_주요요인3', 'SHAP_값1', 'SHAP_값2', 'SHAP_값3']

=== 초안 컬럼 ===
['TOPSIS_우선순위', 'TOPSIS_점수', '학교급', '사고장소', '사고시간', '사고형태', '사고유형', '예측확률_1등급_초고위험', '평균보상액', '사고건수', '위험도_점수', '위험등급', '모델_예측등급', '주요_위험상승요인', '통합_예방전략', '위험도_순위', '위험도_점수_정규화', '모델_최대예측확률', 'SHAP_상승효과_합계', 'SHAP_절대영향_합계', '보상건수', '평균학생수_천명', '빈도율', 'log_빈도율', 'log_평균보상액', '빈도_점수', '심도_점수', '정밀_위험등급', '매트릭스_타겟', 'cluster', '예측확률_2등급_심각위험', '예측확률_3등급_고위험', '학교급_평균_SHAP', '학교급_평균_절대_SHAP', '학교급_SHAP표본수', '사고장소_평균_SHAP', '사고장소_평균_절대_SHAP', '사고장소_SHAP표본수', '사고시간_평균_SHAP', '사고시간_평균_절대_SHAP', '사고시간_SHAP표본수', '사고형태_평균_SHAP', '사고형태_평균_절대_SHAP', '사고형태_SHAP표본수', '최종예측보상액']


In [11]:
import pandas as pd
import numpy as np

# 병합 기준
keys = ["학교급", "사고장소", "사고시간", "사고형태"]

# 1. 문자열 정리
for data in [pred, draft]:
    for col in keys:
        data[col] = (
            data[col]
            .fillna("정보 없음")
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

# 예측보상액 숫자형 변환
pred["최종예측보상액"] = pd.to_numeric(
    pred["최종예측보상액"],
    errors="coerce"
)

draft["최종예측보상액"] = pd.to_numeric(
    draft["최종예측보상액"],
    errors="coerce"
)

print("수정 전 결측:", draft["최종예측보상액"].isna().sum())

# 2. 지역별 예측값을 4개 조건 기준으로 평균
pred_grouped = (
    pred.groupby(keys, as_index=False)["최종예측보상액"]
    .mean()
    .rename(columns={
        "최종예측보상액": "최종예측보상액_재병합"
    })
)

print("예측 조합 수:", len(pred_grouped))

# 3. 초안에 다시 병합
fixed = draft.merge(
    pred_grouped,
    on=keys,
    how="left"
)

# 기존 예측값이 있으면 유지하고,
# 기존 값이 NaN인 경우에만 재병합 예측값으로 채움
fixed["최종예측보상액"] = (
    fixed["최종예측보상액"]
    .fillna(fixed["최종예측보상액_재병합"])
)

fixed = fixed.drop(
    columns=["최종예측보상액_재병합"]
)

print("수정 후 결측:", fixed["최종예측보상액"].isna().sum())
print("채워진 개수:",
      draft["최종예측보상액"].isna().sum()
      - fixed["최종예측보상액"].isna().sum())

수정 전 결측: 7066
예측 조합 수: 9015
수정 후 결측: 7066
채워진 개수: 0


In [12]:
fixed[
    (fixed["학교급"] == "초등학교") &
    (fixed["사고장소"] == "화장실") &
    (fixed["사고시간"] == "과학") &
    (fixed["사고형태"] == "고정된 물체와의 부딪힘")
][[
    "평균보상액",
    "보상건수",
    "최종예측보상액",
    "위험도_점수",
    "위험등급"
]].head()

,평균보상액,보상건수,최종예측보상액,위험도_점수,위험등급
140889,0.0,0,NaN,0.0,3등급_고위험


In [13]:
target = {
    "학교급": "초등학교",
    "사고장소": "화장실",
    "사고시간": "과학",
    "사고형태": "고정된 물체와의 부딪힘"
}

# 예상보상금 파일에서 조건을 하나씩 줄여가며 확인
print("1. 학교급만 일치")
display(
    pred[pred["학교급"].astype(str).str.contains("초등", na=False)]
    [["학교급", "사고장소", "사고시간", "사고형태", "최종예측보상액"]]
    .head(20)
)

print("2. 학교급 + 화장실")
display(
    pred[
        pred["학교급"].astype(str).str.contains("초등", na=False)
        & pred["사고장소"].astype(str).str.contains("화장실", na=False)
    ][["학교급", "사고장소", "사고시간", "사고형태", "최종예측보상액"]]
    .head(30)
)

print("3. 학교급 + 화장실 + 과학 관련")
display(
    pred[
        pred["학교급"].astype(str).str.contains("초등", na=False)
        & pred["사고장소"].astype(str).str.contains("화장실", na=False)
        & pred["사고시간"].astype(str).str.contains("과학", na=False)
    ][["학교급", "사고장소", "사고시간", "사고형태", "최종예측보상액"]]
    .head(30)
)

1. 학교급만 일치


,학교급,사고장소,사고시간,사고형태,최종예측보상액
2,초등학교,일반(교과)교실,쉬는시간,고정된 물체와의 부딪힘,86521.151122
3,초등학교,강당(체육관),체육,넘어짐,157759.042320
9,초등학교,복도,식사시간(간식 포함),넘어짐,214339.315320
11,초등학교,운동장,쉬는시간,1미터 미만의 높이에서 떨어짐,403681.878559
14,초등학교,교통구역(스쿨존 내)-인도,등교,넘어짐,154570.150048
16,초등학교,운동장,체육,고정된 물체와의 부딪힘,140216.986856
17,초등학교,강당(체육관),방과후과정,고정된 물체와의 부딪힘,201109.773118
18,초등학교,복도,방과후과정,고정된 물체와의 부딪힘,185331.222189
19,초등학교,복도,식사시간(간식 포함),사람과의 부딪힘,73700.774550
23,초등학교,운동장,체육,고정된 물체와의 부딪힘,78332.059447


2. 학교급 + 화장실


,학교급,사고장소,사고시간,사고형태,최종예측보상액
411,초등학교,화장실,쉬는시간,넘어짐,128257.861248
1016,초등학교,화장실,자습시간,고정된 물체와의 부딪힘,125111.570144
1052,초등학교,화장실,쉬는시간,고정된 물체와의 부딪힘,191171.817692
1155,초등학교,화장실,쉬는시간,사람과의 부딪힘,136794.576107
1425,초등학교,화장실,쉬는시간,고정된 물체와의 부딪힘,101625.358114
1457,초등학교,화장실,쉬는시간,고정된 물체와의 부딪힘,137218.004858
1478,초등학교,화장실,그 밖의 교육활동 시간,넘어짐,163698.359446
1646,초등학교,화장실,식사시간(간식 포함),고정된 물체와의 부딪힘,127462.785847
1960,초등학교,화장실,쉬는시간,고정된 물체와의 부딪힘,92314.180604
2203,초등학교,화장실,식사시간(간식 포함),"긁힘, 찔림",117570.709194


3. 학교급 + 화장실 + 과학 관련


,학교급,사고장소,사고시간,사고형태,최종예측보상액
25392,초등학교,화장실,과학,"베임, 절단",119743.790066
67842,초등학교,화장실,과학,넘어짐,111977.954616
89649,초등학교,화장실,과학,"베임, 절단",127221.971187
91017,초등학교,화장실,과학,"베임, 절단",88838.873247
108111,초등학교,화장실,과학,"베임, 절단",108269.650147
139328,초등학교,화장실,과학,넘어짐,203868.773801


In [14]:
print("예상보상금 파일 학교급:")
print(pred["학교급"].dropna().unique())

print("\n예상보상금 파일 사고시간 중 과학 포함:")
print(
    pred.loc[
        pred["사고시간"].astype(str).str.contains("과학", na=False),
        "사고시간"
    ].unique()
)

print("\n예상보상금 파일 사고장소 중 화장실 포함:")
print(
    pred.loc[
        pred["사고장소"].astype(str).str.contains("화장실", na=False),
        "사고장소"
    ].unique()
)

예상보상금 파일 학교급:
['고등학교' '초등학교' '중학교' '유치원' '특수학교' '기타학교']

예상보상금 파일 사고시간 중 과학 포함:
['과학' '수학, 과학활동']

예상보상금 파일 사고장소 중 화장실 포함:
['화장실']


In [15]:
keys = ["학교급", "사고장소", "사고시간", "사고형태"]

draft_keys = draft[keys].drop_duplicates()
pred_keys = pred[keys].drop_duplicates()

match_check = draft_keys.merge(
    pred_keys,
    on=keys,
    how="left",
    indicator=True
)

print(match_check["_merge"].value_counts())

_merge
both          7117
left_only     7066
right_only       0
Name: count, dtype: int64


In [16]:
import pandas as pd
import numpy as np

keys = ["학교급", "사고장소", "사고시간", "사고형태"]

# 문자열 정리
for data in [pred, draft]:
    for col in keys:
        data[col] = (
            data[col]
            .fillna("정보 없음")
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

pred["최종예측보상액"] = pd.to_numeric(
    pred["최종예측보상액"],
    errors="coerce"
)

draft["최종예측보상액"] = pd.to_numeric(
    draft["최종예측보상액"],
    errors="coerce"
)

fixed = draft.copy()

# 기존 예측값이 있는 행 표시
fixed["예측보상금_산정기준"] = np.where(
    fixed["최종예측보상액"].notna(),
    "정확한 4개 조건 일치",
    pd.NA
)

print("보완 전 결측:", fixed["최종예측보상액"].isna().sum())


def fill_by_group(target_df, source_df, group_cols, method_name):
    """결측 예측보상액을 지정된 조건 그룹의 중앙값으로 보완"""

    grouped = (
        source_df
        .dropna(subset=["최종예측보상액"])
        .groupby(group_cols, as_index=False)["최종예측보상액"]
        .median()
        .rename(columns={
            "최종예측보상액": "보완예측값"
        })
    )

    merged = target_df[group_cols].merge(
        grouped,
        on=group_cols,
        how="left"
    )

    fill_mask = (
        target_df["최종예측보상액"].isna()
        & merged["보완예측값"].notna()
    )

    target_df.loc[
        fill_mask,
        "최종예측보상액"
    ] = merged.loc[fill_mask, "보완예측값"].to_numpy()

    target_df.loc[
        fill_mask,
        "예측보상금_산정기준"
    ] = method_name

    print(
        f"{method_name}: {fill_mask.sum():,}개 보완, "
        f"남은 결측 {target_df['최종예측보상액'].isna().sum():,}개"
    )

    return target_df


# 1순위: 장소와 사고형태를 유지하고 시간만 완화
fixed = fill_by_group(
    fixed,
    pred,
    ["학교급", "사고장소", "사고형태"],
    "학교급+장소+형태 유사사례 중앙값"
)

# 2순위: 시간과 사고형태를 유지하고 장소 완화
fixed = fill_by_group(
    fixed,
    pred,
    ["학교급", "사고시간", "사고형태"],
    "학교급+시간+형태 유사사례 중앙값"
)

# 3순위: 사고형태 중심
fixed = fill_by_group(
    fixed,
    pred,
    ["학교급", "사고형태"],
    "학교급+형태 유사사례 중앙값"
)

# 4순위: 장소 중심
fixed = fill_by_group(
    fixed,
    pred,
    ["학교급", "사고장소"],
    "학교급+장소 유사사례 중앙값"
)

# 5순위: 학교급
fixed = fill_by_group(
    fixed,
    pred,
    ["학교급"],
    "학교급 전체 중앙값"
)

# 그래도 남은 경우 전체 중앙값
global_median = pred["최종예측보상액"].median()

remaining_mask = fixed["최종예측보상액"].isna()

fixed.loc[
    remaining_mask,
    "최종예측보상액"
] = global_median

fixed.loc[
    remaining_mask,
    "예측보상금_산정기준"
] = "전체 예측보상금 중앙값"

print("\n최종 결측:", fixed["최종예측보상액"].isna().sum())
print("\n산정기준별 개수:")
print(fixed["예측보상금_산정기준"].value_counts())

보완 전 결측: 7066
학교급+장소+형태 유사사례 중앙값: 5,618개 보완, 남은 결측 1,448개
학교급+시간+형태 유사사례 중앙값: 1,084개 보완, 남은 결측 364개
학교급+형태 유사사례 중앙값: 343개 보완, 남은 결측 21개
학교급+장소 유사사례 중앙값: 21개 보완, 남은 결측 0개
학교급 전체 중앙값: 0개 보완, 남은 결측 0개

최종 결측: 0

산정기준별 개수:
예측보상금_산정기준
정확한 4개 조건 일치          135604
학교급+장소+형태 유사사례 중앙값      5618
학교급+시간+형태 유사사례 중앙값      1084
학교급+형태 유사사례 중앙값          343
학교급+장소 유사사례 중앙값           21
Name: count, dtype: int64


In [17]:
fixed[
    (fixed["학교급"] == "초등학교") &
    (fixed["사고장소"] == "화장실") &
    (fixed["사고시간"] == "과학") &
    (fixed["사고형태"] == "고정된 물체와의 부딪힘")
][[
    "학교급",
    "사고장소",
    "사고시간",
    "사고형태",
    "평균보상액",
    "최종예측보상액",
    "예측보상금_산정기준",
    "위험도_점수",
    "위험등급"
]].head()

,학교급,사고장소,사고시간,사고형태,평균보상액,최종예측보상액,예측보상금_산정기준,위험도_점수,위험등급
140889,초등학교,화장실,과학,고정된 물체와의 부딪힘,0.0,113126.308486,학교급+장소+형태 유사사례 중앙값,0.0,3등급_고위험


<문제점>

- 예측을 원래 일부 조합만 했음.

예상보상금_결과.csv를 만든 과정에서, 예측이 가능한 조합만 저장함. 예측파일에는 '초등학교 + 화장실 + 과학 + 넘어짐'가 있는데, '초등학교 + 화장실 + 과학 + 고정된 물체와의 부딪힘'는 아예 없었음. merge를 하면 '최종예측보상액 = NaN'이 됨.

<해결방안>

- 정확한 조합이 없으면, 비슷한 조합 평균 사용

-> 예측 모델 결과가 존재하지 않는 희소 조합의 경우에는 동일 학교급·사고장소·사고형태의 유사사례 중앙 예측값을 활용하여 예측값을 보완.


In [18]:
fixed.to_csv(
    "통합_사고예방_데이터_예측보상금복구.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")

저장 완료


- 위험도_점수, 위험등급은 그대로 둠

- 최종예측보상액이 복구됐으니, CRITIC의 X2, TOPSIS 우선순위만 다시 계산

- X1, X2, X4, CRITIC 가중치, TOPSIS 점수와 순위 다시 계산

In [19]:
import pandas as pd
import numpy as np

# -----------------------------------
# 1. CRITIC-TOPSIS용 데이터 생성
# -----------------------------------

group_cols = ["학교급", "사고장소", "사고시간", "사고형태"]

priority_df = fixed.groupby(group_cols, as_index=False).agg(
    X1_High_Risk_Prob=(
        "예측확률_1등급_초고위험",
        "mean"
    ),
    X2_Predicted_Compensation=(
        "최종예측보상액",
        "mean"
    ),
    X3_Accident_Frequency=(
        "사고건수",
        "max"
    )
)

priority_df["사고유형"] = (
    priority_df["학교급"].astype(str) + "-" +
    priority_df["사고장소"].astype(str) + "-" +
    priority_df["사고시간"].astype(str) + "-" +
    priority_df["사고형태"].astype(str)
)

print("CRITIC-TOPSIS 대상 조합 수:", len(priority_df))


# -----------------------------------
# 2. Min-Max 정규화
# -----------------------------------

criteria_cols = [
    "X1_High_Risk_Prob",
    "X2_Predicted_Compensation",
    "X3_Accident_Frequency"
]

def min_max_normalize(series):
    min_value = series.min()
    max_value = series.max()

    if max_value == min_value:
        return pd.Series(
            np.zeros(len(series)),
            index=series.index
        )

    return (series - min_value) / (max_value - min_value)


normalized_df = priority_df.copy()

normalized_cols = []

for col in criteria_cols:
    normalized_col = f"{col}_Normalized"
    normalized_df[normalized_col] = min_max_normalize(
        normalized_df[col]
    )
    normalized_cols.append(normalized_col)


# -----------------------------------
# 3. CRITIC 가중치 계산
# -----------------------------------

def calculate_critic_weights(df, cols):
    std_dev = df[cols].std()
    corr_matrix = df[cols].corr().fillna(0)

    conflict = (1 - corr_matrix).sum(axis=1)
    information = std_dev * conflict

    if information.sum() == 0:
        return pd.Series(
            [1 / len(cols)] * len(cols),
            index=cols
        )

    return information / information.sum()


critic_weights = calculate_critic_weights(
    normalized_df,
    normalized_cols
)

print("\n=== 새 CRITIC 가중치 ===")

for col, weight in critic_weights.items():
    print(
        col.replace("_Normalized", ""),
        f": {weight:.4f}"
    )


# -----------------------------------
# 4. TOPSIS 계산
# -----------------------------------

def calculate_topsis(df, cols, weights):
    matrix = df[cols].astype(float).copy()

    # 벡터 정규화
    denominator = np.sqrt(
        (matrix ** 2).sum(axis=0)
    ).replace(0, 1)

    normalized_matrix = matrix / denominator

    # CRITIC 가중치 반영
    weighted_matrix = normalized_matrix.multiply(
        weights,
        axis=1
    )

    # 모두 값이 클수록 우선순위가 높은 편익 기준
    positive_ideal = weighted_matrix.max(axis=0)
    negative_ideal = weighted_matrix.min(axis=0)

    positive_distance = np.sqrt(
        ((weighted_matrix - positive_ideal) ** 2)
        .sum(axis=1)
    )

    negative_distance = np.sqrt(
        ((weighted_matrix - negative_ideal) ** 2)
        .sum(axis=1)
    )

    return (
        negative_distance /
        (
            positive_distance
            + negative_distance
            + 1e-10
        )
    )


priority_df["TOPSIS_점수_재계산"] = calculate_topsis(
    normalized_df,
    normalized_cols,
    critic_weights
)

priority_df["TOPSIS_우선순위_재계산"] = (
    priority_df["TOPSIS_점수_재계산"]
    .rank(method="min", ascending=False)
    .astype(int)
)

print("\nTOPSIS 계산 완료")

display(
    priority_df.sort_values(
        "TOPSIS_우선순위_재계산"
    ).head(10)
)


# -----------------------------------
# 5. fixed에 새 TOPSIS 결과 붙이기
# -----------------------------------

fixed_updated = fixed.drop(
    columns=[
        "TOPSIS_점수_재계산",
        "TOPSIS_우선순위_재계산"
    ],
    errors="ignore"
)

fixed_updated = fixed_updated.merge(
    priority_df[
        group_cols + [
            "TOPSIS_점수_재계산",
            "TOPSIS_우선순위_재계산"
        ]
    ],
    on=group_cols,
    how="left"
)

print("\n병합 후 행 수:", len(fixed_updated))
print(
    "TOPSIS 점수 결측:",
    fixed_updated["TOPSIS_점수_재계산"].isna().sum()
)


# -----------------------------------
# 6. 최종 저장
# -----------------------------------

fixed_updated.to_csv(
    "통합_사고예방_데이터_CRITIC_TOPSIS_재계산.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "\n저장 완료:",
    "통합_사고예방_데이터_CRITIC_TOPSIS_재계산.csv"
)

CRITIC-TOPSIS 대상 조합 수: 14183

=== 새 CRITIC 가중치 ===
X1_High_Risk_Prob : 0.7473
X2_Predicted_Compensation : 0.1337
X3_Accident_Frequency : 0.1190

TOPSIS 계산 완료


,학교급,사고장소,사고시간,사고형태,X1_High_Risk_Prob,X2_Predicted_Compensation,X3_Accident_Frequency,사고유형,TOPSIS_점수_재계산,TOPSIS_우선순위_재계산
4801,중학교,강당(체육관),체육,움직이는 물체와의 부딪힘,0.742114,89773.250692,20352,중학교-강당(체육관)-체육-움직이는 물체와의 부딪힘,0.658572,1
7690,중학교,운동장,체육,넘어짐,0.868881,205819.073431,15691,중학교-운동장-체육-넘어짐,0.599378,2
9345,초등학교,강당(체육관),체육,스포츠 활동 중 충격을 가함,0.626487,98675.755526,15919,초등학교-강당(체육관)-체육-스포츠 활동 중 충격을 가함,0.582182,3
9338,초등학교,강당(체육관),체육,넘어짐,0.710160,129468.670219,15551,초등학교-강당(체육관)-체육-넘어짐,0.581997,4
4788,중학교,강당(체육관),체육,고정된 물체와의 부딪힘,0.670301,147158.676198,14664,중학교-강당(체육관)-체육-고정된 물체와의 부딪힘,0.559796,5
4800,중학교,강당(체육관),체육,스포츠 활동 중 충격을 가함,0.846694,133822.231499,13867,중학교-강당(체육관)-체육-스포츠 활동 중 충격을 가함,0.552774,6
7698,중학교,운동장,체육,움직이는 물체와의 부딪힘,0.732467,97357.441969,13849,중학교-운동장-체육-움직이는 물체와의 부딪힘,0.542110,7
9333,초등학교,강당(체육관),체육,고정된 물체와의 부딪힘,0.516896,113381.703826,13228,초등학교-강당(체육관)-체육-고정된 물체와의 부딪힘,0.508869,8
4793,중학교,강당(체육관),체육,넘어짐,0.826392,184370.847606,11694,중학교-강당(체육관)-체육-넘어짐,0.497498,9
7685,중학교,운동장,체육,고정된 물체와의 부딪힘,0.786919,158865.839868,11795,중학교-운동장-체육-고정된 물체와의 부딪힘,0.495616,10



병합 후 행 수: 142670
TOPSIS 점수 결측: 0

저장 완료: 통합_사고예방_데이터_CRITIC_TOPSIS_재계산.csv


- 예상보상금 결측 7066개 -> 0개

- TOPSIS 결측 0개

- CRITIC도 정상

X1 : 0.7473

X2 : 0.1337

X3 : 0.1190


전에 돌렸던 결과

X1: 0.7552

X2: 0.1301

X3: 0.1147


와 거의 동일



In [20]:
# 컬럼 확인

fixed_updated.columns.tolist()

['TOPSIS_우선순위',
 'TOPSIS_점수',
 '학교급',
 '사고장소',
 '사고시간',
 '사고형태',
 '사고유형',
 '예측확률_1등급_초고위험',
 '평균보상액',
 '사고건수',
 '위험도_점수',
 '위험등급',
 '모델_예측등급',
 '주요_위험상승요인',
 '통합_예방전략',
 '위험도_순위',
 '위험도_점수_정규화',
 '모델_최대예측확률',
 'SHAP_상승효과_합계',
 'SHAP_절대영향_합계',
 '보상건수',
 '평균학생수_천명',
 '빈도율',
 'log_빈도율',
 'log_평균보상액',
 '빈도_점수',
 '심도_점수',
 '정밀_위험등급',
 '매트릭스_타겟',
 'cluster',
 '예측확률_2등급_심각위험',
 '예측확률_3등급_고위험',
 '학교급_평균_SHAP',
 '학교급_평균_절대_SHAP',
 '학교급_SHAP표본수',
 '사고장소_평균_SHAP',
 '사고장소_평균_절대_SHAP',
 '사고장소_SHAP표본수',
 '사고시간_평균_SHAP',
 '사고시간_평균_절대_SHAP',
 '사고시간_SHAP표본수',
 '사고형태_평균_SHAP',
 '사고형태_평균_절대_SHAP',
 '사고형태_SHAP표본수',
 '최종예측보상액',
 '예측보상금_산정기준',
 'TOPSIS_점수_재계산',
 'TOPSIS_우선순위_재계산']

In [21]:
# 기존 컬럼 새 결과로 덮어쓰기

fixed_updated["TOPSIS_점수"] = fixed_updated["TOPSIS_점수_재계산"]
fixed_updated["TOPSIS_우선순위"] = fixed_updated["TOPSIS_우선순위_재계산"]

# 재계산 컬럼 삭제
fixed_updated.drop(
    columns=[
        "TOPSIS_점수_재계산",
        "TOPSIS_우선순위_재계산"
    ],
    inplace=True
)

fixed_updated.to_csv(
    "통합_사고예방_데이터_최종.csv",
    index=False,
    encoding="utf-8-sig"
)

print("최종 CSV 저장 완료")

최종 CSV 저장 완료


In [22]:
# 위험도_점수=0, 위험등급=3등급_고위험 같은 경우가 남아 있을 가능성 확인

fixed_updated[
    (fixed_updated["학교급"]=="초등학교") &
    (fixed_updated["사고장소"]=="화장실") &
    (fixed_updated["사고시간"]=="과학") &
    (fixed_updated["사고형태"]=="고정된 물체와의 부딪힘")
][[
    "최종예측보상액",
    "위험도_점수",
    "위험등급",
    "TOPSIS_점수",
    "TOPSIS_우선순위"
]]

,최종예측보상액,위험도_점수,위험등급,TOPSIS_점수,TOPSIS_우선순위
140889,113126.308486,0.0,3등급_고위험,0.035103,12892


In [23]:
fixed_updated[
    (fixed_updated["학교급"] == "초등학교") &
    (fixed_updated["사고장소"] == "화장실") &
    (fixed_updated["사고시간"] == "과학") &
    (fixed_updated["사고형태"] == "고정된 물체와의 부딪힘")
][[
    "위험도_점수",
    "위험등급",
    "모델_예측등급",
    "예측확률_1등급_초고위험",
    "예측확률_2등급_심각위험",
    "예측확률_3등급_고위험",
    "모델_최대예측확률",
    "빈도율",
    "평균보상액",
    "최종예측보상액",
    "TOPSIS_점수",
    "TOPSIS_우선순위"
]]

,위험도_점수,위험등급,모델_예측등급,예측확률_1등급_초고위험,예측확률_2등급_심각위험,예측확률_3등급_고위험,모델_최대예측확률,빈도율,평균보상액,최종예측보상액,TOPSIS_점수,TOPSIS_우선순위
140889,0.0,3등급_고위험,3등급_고위험,0.111938,0.348895,0.539166,0.539166,0.000782,0.0,113126.308486,0.035103,12892


<문제점>

- 서로 다른 두 시스템의 결과를 동시에 보여줌.

- 위험도_점수는 과거 실제 사고 기록만으로 계산하고, 위험등급은 AI 분류모델이 예측한 결과.

-> 과거에는 거의 발생하지 않았지만, 모델은 다른 특징을 보고 고위험일 가능성이 높다고 판단.

In [28]:
print("행 수:", len(fixed_updated))
print("예측보상금 결측:", fixed_updated["최종예측보상액"].isna().sum())
print("TOPSIS 점수 결측:", fixed_updated["TOPSIS_점수"].isna().sum())
print("TOPSIS 순위 결측:", fixed_updated["TOPSIS_우선순위"].isna().sum())

행 수: 142670
예측보상금 결측: 0
TOPSIS 점수 결측: 0
TOPSIS 순위 결측: 0


In [29]:
# Streamlit에 업로드하기위해 github에 저장하는 과정에서, 압축 과정

import pandas as pd

# 이미 fixed_updated가 메모리에 있으면 그대로 사용
df = fixed_updated.copy()

keys = ["학교급", "사고장소", "사고시간", "사고형태"]

print("원본 행 수:", len(df))
print("고유 조합 수:", df[keys].drop_duplicates().shape[0])


원본 행 수: 142670
고유 조합 수: 14183


In [30]:
check_cols = [
    "사고건수",
    "최종예측보상액",
    "모델_예측등급",
    "TOPSIS_점수",
    "TOPSIS_우선순위"
]

for col in check_cols:
    max_unique = df.groupby(keys, dropna=False)[col].nunique(dropna=False).max()
    print(f"{col} - 조합 내 최대 고유값 수:", max_unique)

사고건수 - 조합 내 최대 고유값 수: 1
최종예측보상액 - 조합 내 최대 고유값 수: 2470
모델_예측등급 - 조합 내 최대 고유값 수: 1
TOPSIS_점수 - 조합 내 최대 고유값 수: 1
TOPSIS_우선순위 - 조합 내 최대 고유값 수: 1


In [31]:
import gzip
import shutil

with open("통합_사고예방_데이터_최종.csv", "rb") as f_in:
    with gzip.open("통합_사고예방_데이터_최종.csv.gz", "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

print("압축 완료")

압축 완료


In [32]:
import os

size = os.path.getsize("통합_사고예방_데이터_최종.csv.gz") / 1024 / 1024
print(f"{size:.2f} MB")

5.22 MB


In [33]:
df = pd.read_csv(
    "통합_사고예방_데이터_최종.csv.gz",
    compression="gzip",
    encoding="utf-8-sig"
)